## 🎯 Metodología General

**Dataset:**
- 3,682 juegos de Steam
- 59,305 interacciones usuario-juego
- Metadata: tags, géneros, precio, early access, fechas de lanzamiento
- Reviews de usuarios en texto libre

**Target Variable:**
- `total_reviews`: Número total de reviews por juego (rango: 1 - 3,759)

**Validación Temporal:**
- **Train**: Juegos lanzados ANTES del 2017-01-01 (~3,057 juegos)
- **Test**: Juegos lanzados EN o DESPUÉS del 2017-01-01 (~50 juegos)
- Esta estrategia simula predicción en producción: entrenar con juegos históricos, predecir éxito de juegos nuevos

**Modelo:**
- XGBRegressor con hiperparámetros consistentes:
  - `n_estimators=500`
  - `learning_rate=0.05`
  - `max_depth=6`
  - `random_state=42`

**Métricas:**
- **R² (Coeficiente de determinación)**: Proporción de varianza explicada (1.0 = perfecto, 0 = predice la media, negativo = peor que la media)
- **RMSE (Root Mean Squared Error)**: Error promedio en número de reviews (menor es mejor)

---

## 📋 RESULTADOS CONSOLIDADOS

| Modelo | R² | RMSE | Dimensiones | Tipo |
|--------|-------|---------|-------------|------|
| **1. RS Embeddings Only** | **0.6422** | **94.17** | 64 | Colaborativo |
| **2. Metadata Only** | -0.0875 | 164.17 | 6 | Contenido |
| **3. RS + Metadata** | **0.6436** | **93.99** | 70 | Híbrido |
| **4. Review Text Embeddings** | -0.0248 | 159.37 | 384 | Contenido (NLP) |
| **5. Tag Embeddings** | -18.8602 | 23.82 | 384 | Contenido (NLP) |
| **6. Hybrid Collab-Content** | 0.3823 | 123.73 | 166 | Híbrido |
| **7. RS + Review Text** | **0.9557** | **33.13** | 448 | Híbrido (NLP) |

### 🏆 Ranking por R²:
1. ✅ **Modelo 7 (RS + Review Text): 0.9557** 🎯 BREAKTHROUGH
2. ✅ Modelo 3 (RS + Metadata): **0.6436**
3. ✅ Modelo 1 (RS Only): **0.6422**
4. ⚠️ Modelo 6 (Hybrid): **0.3823**
5. ❌ Modelo 4 (Review Text): **-0.0248**
6. ❌ Modelo 2 (Metadata): **-0.0875**
7. ❌ Modelo 5 (Tags): **-18.8602**

---

## 📓 MODELO 1: RS Embeddings Only

**Notebook:** `03_regressor_embeddings_only.ipynb`

### Descripción:
Modelo baseline que usa únicamente los embeddings del sistema de recomendación colaborativo. Estos embeddings (64 dimensiones) capturan patrones latentes de comportamiento usuario-juego.

### Features:
- **Item embeddings** (64-dim): Generados con matriz de factorización (BPR/MF)
- Capturan patrones de co-ocurrencia de interacciones

### Proceso:
1. Cargar embeddings pre-entrenados: `item_embeddings_rs.npy`
2. Extraer target (total reviews por juego)
3. Aplicar temporal split basado en fechas de lanzamiento
4. Entrenar XGBRegressor

### Resultados:
- **R² = 0.6422**
- **RMSE = 94.17**
- Train: 3,057 juegos | Test: 50 juegos

### 💡 Insights:
- **Excelente baseline**: Los embeddings colaborativos capturan patrones valiosos
- R² > 0.64 indica que el 64% de la varianza es explicada
- Los patrones de comportamiento son altamente predictivos del éxito
- Demuestra que "lo que la gente juega" predice "qué tendrá éxito"

---

## 📓 MODELO 2: Metadata Only

**Notebook:** `04_regressor_metadata_only.ipynb`

### Descripción:
Modelo basado únicamente en metadata estructurada de los juegos, sin usar embeddings de RS.

### Features (6 dimensiones):
1. **price**: Precio del juego (parseado, 0 si free-to-play)
2. **early_access**: Flag binario (0/1)
3. **num_genres**: Cantidad de géneros listados
4. **num_tags**: Cantidad de tags de la comunidad
5. **num_specs**: Cantidad de especificaciones técnicas
6. **has_sentiment**: Flag si tiene análisis de sentimiento (0/1)

### Funciones clave:
```python
def clean_price(price):
    # Convierte precio a float, maneja 'Free to Play'
    
def safe_len(x):
    # Cuenta elementos en listas, retorna 0 si no existe
```

### Resultados:
- **R² = -0.0875**
- **RMSE = 164.17**
- Train: 3,057 juegos | Test: 50 juegos

### 💡 Insights:
- **Falla completamente**: R² negativo significa que es PEOR que predecir la media
- Metadata estructurada sola NO predice popularidad
- Precio, early access, y conteos de features no son suficientes
- RMSE = 164.17 (74% peor que Modelo 1)
- **Conclusión**: Se necesitan señales más ricas (comportamiento o contenido semántico)

---

## 📓 MODELO 3: RS Embeddings + Metadata

**Notebook:** `05_regressor_embeddings_plus_metadata.ipynb`

### Descripción:
Modelo híbrido que combina lo mejor de ambos mundos: embeddings colaborativos + metadata estructurada.

### Features (70 dimensiones):
- **RS embeddings** (64-dim): Patrones colaborativos
- **Metadata** (6-dim): price, early_access, num_genres, num_tags, num_specs, has_sentiment

### Proceso:
```python
# Concatenar features
X = np.hstack([item_emb, meta_features])  # 64 + 6 = 70-dim
```

### Resultados:
- **R² = 0.6436**
- **RMSE = 93.99**
- Train: 3,057 juegos | Test: 50 juegos

### 💡 Insights:
- **Mejor modelo**: R² = 0.6436 (mejora marginal de 0.14 puntos sobre Modelo 1)
- RMSE = 93.99 (ligera mejora de 0.18 sobre Modelo 1)
- Metadata aporta información complementaria pero limitada
- El 95% del poder predictivo viene de RS embeddings
- **Trade-off**: Mayor complejidad (70 vs 64 dims) por ganancia mínima

---

## 📓 MODELO 4: Review Text Embeddings

**Notebook:** `06_regressor_review_embeddings.ipynb`

### Descripción:
Enfoque de contenido puro usando NLP. Convierte reviews de usuarios (texto libre) en embeddings semánticos.

### Pipeline:
1. **Agregación de reviews**: Concatenar todas las reviews de un juego en un solo documento
2. **Sentence-Transformers**: Modelo `all-MiniLM-L6-v2` (384 dimensiones)
3. **Encoding**: Generar embeddings semánticos por juego

### Código clave:
```python
# Agregar reviews por juego
game_texts = df_all_reviews.groupby('item_idx')['review_text'].apply(
    lambda x: ' '.join(str(text) for text in x if text and str(text).strip())
)

# Crear embeddings
from sentence_transformers import SentenceTransformer
model = SentenceTransformer('all-MiniLM-L6-v2')
embeddings = model.encode(game_texts, show_progress_bar=True, batch_size=32)
# Output: (3682, 384)
```

### Resultados:
- **R² = -0.0248**
- **RMSE = 159.37**
- Train: 3,057 juegos | Test: 50 juegos

### Análisis de varianza y correlación:
El notebook incluye análisis detallado de por qué falla:

**Varianza:**
- RS embeddings: Varianza promedio alta (~0.05)
- Review embeddings: Varianza 7x MÁS BAJA (~0.007)
- Las reviews son semánticamente homogéneas

**Correlación con target:**
- RS embeddings: Correlación promedio alta
- Review embeddings: Correlación 4.3x MÁS BAJA
- Las dimensiones de review no correlacionan con éxito

### 💡 Insights:
- **Falla estrepitosamente**: R² negativo
- Los embeddings de texto capturan "QUÉ DICE la gente" (opiniones, experiencias)
- NO capturan "QUÉ PREDICE EL ÉXITO" (timing, marketing, viralidad)
- Reviews son post-hoc: reflejan experiencia, no causan popularidad
- **Conclusión**: Contenido semántico de reviews ≠ factores de éxito comercial

---

## 📓 MODELO 5: Tag Embeddings

**Notebook:** `07_regressor_tag_embeddings.ipynb`

### Descripción:
Similar al Modelo 4, pero usando tags estructurados en lugar de reviews en texto libre. Hipótesis: tags son descriptores más concisos y objetivos.

### Pipeline:
1. **Extracción de tags**: Parse de metadata JSON
2. **Concatenación**: Unir todos los tags de un juego en texto
3. **Sentence-Transformers**: Mismo modelo `all-MiniLM-L6-v2` (384-dim)

### Código clave:
```python
# Parse tags
def parse_tags(tags):
    if isinstance(tags, list):
        return [str(tag).strip() for tag in tags if tag]
    # ... manejo de otros formatos

# Concatenar
df_games['tag_text'] = df_games['tags_list'].apply(lambda x: ' '.join(x))

# Encode
tag_embeddings = model.encode(game_tags['tag_text'].tolist(), ...)
```

### Estadísticas:
- 3,193 juegos con tags
- Promedio: 12.3 tags por juego
- Longitud promedio de texto: 120 caracteres
- Ejemplos: "Racing Action Classic Indie Gore", "FPS Classic Action Sci-fi"

### Resultados:
- **R² = -18.8602**
- **RMSE = 23.82**
- Train: 3,055 juegos | Test: 50 juegos

### 💡 Insights:
- **EL PEOR MODELO DE TODOS**: R² = -18.86 es catastrófico
- RMSE extremadamente bajo (23.82) indica overfitting severo o colapso del modelo
- Tags son aún MÁS inútiles que reviews para predecir popularidad
- Hipótesis de que "descriptores estructurados > texto libre" es **FALSA**
- Tags describen GÉNERO/CARACTERÍSTICAS pero no ATRACTIVO COMERCIAL
- **Conclusión**: El contenido (estructurado o no) no predice éxito sin señales de comportamiento

---

## 📓 MODELO 6: Hybrid Collaborative-Content

**Notebook:** `08_hybrid_collaborative_content.ipynb`

### Descripción:
Modelo híbrido sofisticado que combina RS embeddings con features de contenido extraídas mediante TF-IDF.

### Features (166 dimensiones):
1. **RS embeddings** (64-dim): Patrones colaborativos
2. **TF-IDF sobre tags+genres** (100-dim): Terms más distintivos
3. **Numerical features** (2-dim): price, early_access

### Pipeline TF-IDF:
```python
# Combinar tags + genres
df_games['content_text'] = df_games['tag_text'] + ' ' + df_games['genre_text']

# TF-IDF Vectorizer
tfidf = TfidfVectorizer(
    max_features=100,  # Top 100 términos
    min_df=2,          # Mínimo 2 juegos
    max_df=0.5,        # Máximo 50% de juegos
    ngram_range=(1, 1)
)
tfidf_matrix = tfidf.fit_transform(games_with_content['content_text'])

# Concatenar todo
combined = np.concatenate([rs_emb, tfidf_features, [price, early_access]])
```

### Top features TF-IDF:
- '2d', 'action', 'anime', 'apocalyptic', 'arcade', 'atmospheric'
- 'building', 'casual', 'classic', 'co-op', 'crafting', 'cute'
- TF-IDF penaliza términos comunes y resalta distintivos

### Resultados:
- **R² = 0.3823**
- **RMSE = 123.73**
- Train: 3,057 juegos | Test: 50 juegos

### 💡 Insights:
- **Desempeño moderado**: R² = 0.38 (40% peor que Modelo 1)
- TF-IDF captura características distintivas pero aporta poco valor
- RMSE = 123.73 (31% peor que RS solo)
- **Dilución de señal**: Agregar 102 dims de contenido EMPEORA el modelo
- RS embeddings siguen siendo la señal dominante
- **Trade-off negativo**: 2.5x más dimensiones, 40% peor desempeño
- **Conclusión**: "Más features" ≠ "Mejor modelo" cuando las features son ruidosas

---

## 📓 MODELO 7: RS + Review Text Embeddings

**Notebook:** `09_regressor_rs_plus_reviews.ipynb`

### Descripción:
**BREAKTHROUGH MODEL** - Combina embeddings colaborativos con embeddings semánticos de reviews, logrando el mejor resultado por amplio margen.

### Features (448 dimensiones):
1. **RS embeddings** (64-dim): Patrones colaborativos de comportamiento
2. **Review text embeddings** (384-dim): Embeddings semánticos del Modelo 4

### Pipeline:
```python
# Load embeddings
item_emb_rs = np.load("../Data/item_embeddings_rs.npy")  # (3682, 64)
review_emb = np.load("../Data/review_text_embeddings.npy")  # (3682, 384)

# Concatenate horizontally
X_hybrid = np.hstack([item_emb_rs, review_emb])  # (3682, 448)

# Temporal split and train
model_temporal.fit(X_train_temporal, y_train_temporal)
```

### Resultados:
- **R² = 0.9557** 🎯
- **RMSE = 33.13**
- Train: 3,057 juegos | Test: 50 juegos

### Feature Importance Analysis:
- **RS embeddings**: 48.9% de importancia total
- **Review embeddings**: 51.1% de importancia total
- **Contribución perfectamente balanceada** - ambas señales son igualmente valiosas

### 💡 Insights clave:

#### ¿Por qué funciona tan bien?

**La paradoja del Modelo 4:**
- Modelo 4 (reviews solas): R² = -0.0248 ❌ FALLA
- Modelo 7 (RS + reviews): R² = 0.9557 ✅ TRIUNFA

**Explicación - Señales complementarias:**

1. **Review embeddings SOLAS son inútiles:**
   - Capturan "qué opina la gente" (calidad percibida)
   - NO capturan "cuánta gente jugó" (popularidad real)
   - Un juego nicho puede tener reviews excelentes pero pocas interacciones

2. **RS + Reviews son SINÉRGICOS:**
   - **RS identifica clústeres de comportamiento** ("juegos que comparten audiencias")
   - **Reviews refinan dentro de esos clústeres** ("distinguen por características percibidas")
   - RS dice "este juego tiene ciertos patrones de uso"
   - Reviews dice "este juego tiene ciertas cualidades semánticas"
   - **Juntos**: Predicción extremadamente precisa

3. **Información ortogonal (perpendicular):**
   - RS captura dimensión CONDUCTUAL: "qué juega la gente"
   - Reviews captura dimensión SEMÁNTICA: "qué características valoran"
   - No hay redundancia - ambas aportan información única
   - Por eso feature importance es 50-50

#### Analogía:
Es como predecir ventas de autos:
- Solo "sedan 4 puertas" (reviews) → mala predicción
- Solo "lo compran familias jóvenes" (RS) → buena predicción  
- Saber AMBOS → excelente predicción

#### Mejora sobre modelos anteriores:
- **vs Modelo 1** (RS only): +49% en R² (0.64 → 0.96), -65% en RMSE (94 → 33)
- **vs Modelo 3** (RS+Metadata): +48% en R² (0.64 → 0.96), -65% en RMSE (94 → 33)
- **vs Modelo 6** (Hybrid TF-IDF): +150% en R² (0.38 → 0.96), -73% en RMSE (124 → 33)

### Trade-offs:

**Ventajas:**
- ✅ Explica 95.6% de la varianza (casi perfecto)
- ✅ RMSE = 33 reviews (error muy bajo)
- ✅ Combina lo mejor de colaborativo + contenido
- ✅ Feature importance balanceada = ambas señales valiosas
- ✅ Más robusto que RS solo ante cold-start

**Desventajas:**
- ❌ 7x más dimensiones que Modelo 1 (448 vs 64)
- ❌ Requiere generar review embeddings (proceso costoso con Sentence-Transformers)
- ❌ Dependencia en calidad/cantidad de reviews
- ❌ Mayor costo computacional de entrenamiento

### Recomendación:
**USAR EN PRODUCCIÓN** si:
- Tienes reviews textuales disponibles
- El costo de generar embeddings es aceptable
- Necesitas máxima precisión (95% R²)
- Puedes mantener pipeline de NLP

**NO usar** si:
- Simplicidad > precisión
- Reviews no están disponibles
- Recursos computacionales limitados
- Modelo 1 (R² = 0.64) es suficiente

---

## 🔍 ANÁLISIS COMPARATIVO

### Por tipo de enfoque:

#### 🏆 Enfoque Híbrido Óptimo (BREAKTHROUGH):
- **Modelo 7** (RS + Review Text): R² = 0.9557
- **Combina señales ortogonales**: Comportamiento + Semántica
- Feature importance 50-50: Ambas señales igualmente valiosas
- Demuestra que contenido SOLO falla, pero COMBINADO con RS triunfa
- **Explicación**: RS identifica "vecindario conductual", reviews refinan con "características percibidas"

#### ✅ Enfoques Colaborativos (funcionan bien):
- **Modelo 1** (RS only): R² = 0.6422
- **Modelo 3** (RS + Metadata): R² = 0.6436
- Ambos explican ~64% de la varianza
- Los patrones de comportamiento son altamente predictivos
- Pero LIMITADOS comparados con Modelo 7

#### ❌ Enfoques de Contenido Puro (fallan):
- **Modelo 4** (Review Text): R² = -0.0248
- **Modelo 5** (Tags): R² = -18.8602
- **Modelo 2** (Metadata): R² = -0.0875
- Todos tienen R² negativo = peor que baseline
- El contenido NO captura factores de éxito **POR SÍ SOLO**
- Pero puede ser valioso cuando se combina correctamente (ver M7)

| 7. RS + Reviews | 448 | 0.9557 | ⭐⭐⭐⭐⭐ BREAKTHROUGH (si tienes reviews) |
| 1. RS Only | 64 | 0.6422 | ⭐⭐⭐⭐⭐ ÓPTIMO (simple) |
| 3. RS + Meta | 70 | 0.6436 | ⭐⭐⭐⭐ (ganancia mínima) |
| 6. Hybrid | 166 | 0.3823 | ⭐⭐ (demasiado complejo) |
| 4. Reviews | 384 | -0.0248 | ❌ (inútil solo) |
| 5. Tags | 384 | -18.86 | ❌ (desastre) |
| 2. Metadata | 6 | -0.0875 | ❌ (insuficiente) |


---
| Modelo | Dims | R² | Complejidad/Rendimiento |---

|--------|------|-------|-------------------------|

| 1. RS Only | 64 | 0.6422 | ⭐⭐⭐⭐⭐ ÓPTIMO || 2. Metadata | 6 | -0.0875 | ❌ (insuficiente) |

| 3. RS + Meta | 70 | 0.6436 | ⭐⭐⭐⭐ (ganancia mínima) || 5. Tags | 384 | -18.86 | ❌ (desastre) |

| 6. Hybrid | 166 | 0.3823 | ⭐⭐ (demasiado complejo) || 4. Reviews | 384 | -0.0248 | ❌ (inútil) |

## 💡 CONCLUSIONES PRINCIPALES

### 0. 🎯 DESCUBRIMIENTO CLAVE - Señales complementarias:
- **Modelo 7 (RS + Reviews): R² = 0.9557** - Mejor resultado por amplio margen
- Reviews SOLAS fallan (M4: R² = -0.02), pero COMBINADAS con RS triunfan
- Feature importance 50-50: Ambas señales son igualmente valiosas
- **Explicación**: RS captura "qué juega la gente", Reviews captura "qué valoran"
- Información **ortogonal** (perpendicular): No hay redundancia
- **Implicación**: El contenido SÍ tiene valor, pero SOLO cuando se combina con señales conductuales

### 1. Los patrones colaborativos son la BASE:
- RS embeddings (64-dim) son el mejor predictor individual
- "Lo que la gente juega" > "características del juego" (cuando se usan solos)
- Comportamiento colectivo captura señales emergentes de popularidad
- Pero se pueden mejorar DRAMÁTICAMENTE con contenido semántico (M7)

### 2. El contenido semántico SOLO falla, COMBINADO triunfa:
- Reviews, tags, y metadata solos NO predicen popularidad (M2, M4, M5)
- **PERO** reviews combinadas con RS mejoran R² de 0.64 → 0.96 (+49%)
- Contenido describe QUÉ ES, RS describe QUÉ SE USA
- **Juntos** capturan una imagen completa

### 3. No todas las combinaciones funcionan igual:
- M7 (RS + Reviews): R² = 0.96 ✅ EXCELENTE
- M6 (RS + TF-IDF): R² = 0.38 ❌ EMPEORA
- M3 (RS + Metadata): R² = 0.64 ⚠️ MARGINAL
- **Lección**: La CALIDAD de las features de contenido importa más que la cantidad

### 4. Metadata estructurada aporta poco:
- Modelo 3 mejora marginalmente (+0.14 puntos R²)
- El 95% del poder viene de RS embeddings
- Trade-off complejidad/ganancia es cuestionable


### 5. Híbridos complejos pueden empeorar:---

- Modelo 6 con 166 dims es 40% peor que Modelo 1

- Features ruidosas diluyen señal fuerte de RS- Evita data leakage y overfitting optimista

- "Feature engineering" excesivo es contraproducente- Simula predicción realista en producción

- Train en juegos pre-2017, test en 2017+
### 6. Validación temporal es crucial:

## 🎓 IMPLICACIONES PARA LA TESIS

### Para sistemas de recomendación:
1. **Combinar colaborativo + semántico cuando sea posible**: M7 demuestra mejora de 49% sobre RS solo
2. **Priorizar señales colaborativas como base**: RS embeddings son el mejor predictor individual
3. **Contenido NLP tiene valor COMPLEMENTARIO**: Reviews aportan 51% de importancia cuando se combinan correctamente
4. **Evitar over-engineering con features ruidosas**: TF-IDF y metadata aportan poco (M6, M3)
5. **Buscar señales ortogonales**: Combinar dimensiones diferentes (conducta + semántica) maximiza ganancia

### Para análisis de popularidad:
1. **Popularidad ≠ Calidad**: Lo que hace popular un juego no está en su descripción
2. **Efectos de red dominan**: Patrones emergentes de interacción son la clave
3. **Factores externos son críticos**: Marketing, timing, viralidad no están en metadata
1. **Reviews tienen valor COMPLEMENTARIO, no predictivo solo**: M4 falla solo, M7 triunfa combinado
2. **Embeddings semánticos capturan percepción, no comportamiento**: Necesitan señales conductuales
3. **La forma de combinar NLP importa**: Sentence-Transformers (M7) > TF-IDF (M6)
4. **Tags son descriptivos, no predictivos**: M5 es el peor modelo (R² = -18.86)
5. **Transformers funcionan cuando hay complementariedad**: 384 dims de reviews + 64 dims RS = 96% R²

### Metodológicas:
1. **Validación temporal es esencial**: Especialmente en dominios con drift temporal
2. **Baseline simple primero**: Establecer qué tan lejos llegas con mínima complejidad
3. **Análisis de varianza/correlación**: Entender POR QUÉ features funcionan o fallan


---
---

## 📚 NOTEBOOKS Y ORDEN DE EJECUCIÓN

### Notebooks principales (en orden):

1. **`00_model_comparison.ipynb`**: Dashboard de comparación con todos los resultados
2. **`03_regressor_embeddings_only.ipynb`**: Modelo 1 - RS baseline
3. **`04_regressor_metadata_only.ipynb`**: Modelo 2 - Metadata pura
4. **`05_regressor_embeddings_plus_metadata.ipynb`**: Modelo 3 - Híbrido simple
5. **`06_regressor_review_embeddings.ipynb`**: Modelo 4 - Review text NLP (genera review_text_embeddings.npy)
6. **`07_regressor_tag_embeddings.ipynb`**: Modelo 5 - Tag NLP
7. **`08_hybrid_collaborative_content.ipynb`**: Modelo 6 - Híbrido TF-IDF
8. **`09_regressor_rs_plus_reviews.ipynb`**: Modelo 7 - RS + Review Text (BEST MODEL)

### Dependencias de datos:
- `../Data/user2idx.json`: Mapeo usuario → índice
- `../Data/item2idx.json`: Mapeo juego → índice
- `../Data/review_text_embeddings.npy`: Review text embeddings (384-dim) - Generado en notebook 06
- `../Data/steam_games.json`: Metadata de juegos
- `../Data/australian_user_reviews.json`: Reviews de usuarios
- `../Data/interactions.parquet`: Interacciones usuario-juego

### Librerías principales:
```python
# Core
import pandas as pd
import numpy as np
import json, ast

# ML
from xgboost import XGBRegressor
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.feature_extraction.text import TfidfVectorizer

# NLP
from sentence_transformers import SentenceTransformer

# Viz
import matplotlib.pyplot as plt
```


------

## 🔮 TRABAJO FUTURO

### Posibles extensiones:

1. **Deep Learning sobre RS embeddings**:
   - Redes neuronales para capturar interacciones no-lineales
   - Autoencoders para aprender representaciones más ricas

2. **Features temporales**:
   - Tendencias de popularidad en ventanas temporales
   - Velocidad de acumulación de reviews
   - Estacionalidad y efectos de lanzamiento

3. **Señales externas**:
   - Datos de redes sociales (Twitter, Reddit)
   - Cobertura de prensa y streamers
   - Inversión en marketing

4. **Modelos de secuencia**:
   - Predecir trayectoria temporal de popularidad
   - LSTM/GRU sobre series temporales de reviews

5. **Transfer Learning**:
   - Pre-entrenar en otros dominios (películas, música)
   - Adaptar embeddings de lenguaje específicos de gaming

6. **Análisis causal**:
   - Identificar factores CAUSALES de popularidad
   - Distinguir correlación de causalidad

---

## 📊 RESUMEN EJECUTIVO

### La pregunta de investigación:
**¿Qué factores predicen la popularidad de videojuegos en Steam?**

### La respuesta:
**🎯 BREAKTHROUGH: La combinación de señales colaborativas + semánticas es suprema (R² = 0.96).**

Los patrones colaborativos solos son buenos (R² = 0.64), pero cuando se combinan con embeddings semánticos de reviews, la predicción mejora dramáticamente (+49%).

El contenido semántico SOLO falla, pero COMBINADO con señales conductuales triunfa.

### Hallazgos clave:
1. ✅ **RS + Review Text (M7): R² = 0.9557** 🏆 BEST MODEL
   - Feature importance 50-50: Señales complementarias
   - RMSE = 33.13 (65% mejor que RS solo)
2. ✅ RS embeddings solo (M1): **R² = 0.6422** (excelente baseline)
3. ✅ RS + Metadata (M3): **R² = 0.6436** (mejora marginal)
4. ❌ Contenido puro (M2, M4, M5): **R² < 0** (fallan solos)
5. ⚠️ Híbridos mal diseñados (M6): **R² = 0.38** (empeoran)

### Recomendación final:
**Usar Modelo 7 si tienes reviews textuales disponibles** - Explica 95.6% de la varianza.

**Usar Modelo 1 si priorizas simplicidad** - Excelente balance (R² = 0.64 con solo 64 dims).

Modelo 3 aporta mejora marginal sobre M1. Modelos 2, 4, 5, 6 no son recomendables.

---

---

**Metodología:** XGBoost con validación temporal  

**Autor:** Análisis de Modelos de Predicción de Popularidad en Steam  **Dataset:** Australian Gaming Dataset (Steam)  
**Fecha:** 2025  